# UC Question Sprint



In [5]:
import pandas as pd

bay = pd.read_csv("bay_area_modeling_table.csv")
eth = pd.read_csv("uc_admissions_summary_by_ethnicity.csv")
disc = pd.read_csv("uc_freshman_admission_by_discipline.csv")
transfer = pd.read_csv("uc_transfer_admission_by_major.csv")



## Q1. Fall 2025: average number of UC campuses per applicant
Uses `uc_admissions_summary_by_ethnicity.csv`. Each campus row = applicants to that campus; the `Systemwide` row = distinct (unduplicated) applicants. Sum of per-campus applicants ÷ systemwide distinct applicants = average campuses applied to.

In [6]:
d25 = eth[(eth.fall_term == 2025) & (eth.count_type == 'App')]

for level in ['freshman', 'transfer']:
    sub = d25[d25.entrant_level == level]
    per_campus_sum = sub[sub.campus != 'Systemwide']['n'].sum()
    systemwide = sub[sub.campus == 'Systemwide']['n'].sum()
    avg = per_campus_sum / systemwide
    print(f"{level}: {avg:.2f} campuses/applicant  (per-campus sum={per_campus_sum}, systemwide={systemwide})")

# combined freshman + transfer
per_campus_all = d25[d25.campus != 'Systemwide']['n'].sum()
systemwide_all = d25[d25.campus == 'Systemwide']['n'].sum()
print(f"combined: {per_campus_all/systemwide_all:.2f} campuses/applicant")

freshman: 4.54 campuses/applicant  (per-campus sum=932623, systemwide=205389)
transfer: 3.75 campuses/applicant  (per-campus sum=168299, systemwide=44900)
combined: 4.40 campuses/applicant


## Q2. Fall 2025 admit rate at UCLA for Bay Area public high school applicants
Uses `bay_area_modeling_table.csv`, campus = `Los Angeles`, `school_type` containing "Public".

In [4]:
ucla25 = bay[(bay.campus == 'Los Angeles') & (bay.fall_term == 2025)]
pub = ucla25[ucla25.school_type.astype(str).str.contains('Public', na=False)]

tot_app = pub['applicants'].sum()
tot_adm = pub['admits'].sum()
print(f"Bay Area public HS -> UCLA, fall 2025: {tot_adm:.0f} admits / {tot_app:.0f} applicants = {tot_adm/tot_app:.4f}")

# for comparison: UCLA overall (all applicants, all origins), from the discipline file
ucla_all = disc[(disc.campus == 'Los Angeles') & (disc.fall_term == 2025) & (disc.broad_discipline == 'All disciplines')]
print(ucla_all[['applicants', 'admits', 'admit_rate']])

Bay Area public HS -> UCLA, fall 2025: 1487 admits / 17965 applicants = 0.0828
    applicants  admits  admit_rate
36      145060   13659        0.09


## Q3. Fall 2025: which UC campus's Computer Science admit rate lags its overall admit rate the most?
Uses `uc_freshman_admission_by_discipline.csv`, comparing each campus's `Computer Science` row to its `All disciplines` row.

In [7]:
d25disc = disc[disc.fall_term == 2025]

cs = d25disc[d25disc.broad_discipline == 'Computer Science'][['campus','applicants','admits','admit_rate']]
cs = cs.rename(columns={'applicants':'cs_app','admits':'cs_adm','admit_rate':'cs_rate'})

allc = d25disc[d25disc.broad_discipline == 'All disciplines'][['campus','applicants','admits','admit_rate']]
allc = allc.rename(columns={'applicants':'all_app','admits':'all_adm','admit_rate':'all_rate'})

m = cs.merge(allc, on='campus')
m['gap_pts'] = (m['all_rate'] - m['cs_rate']) * 100
m = m.sort_values('gap_pts', ascending=False)
print(m.to_string(index=False))
print("\nBiggest CS penalty:", m.iloc[0]['campus'], f"({m.iloc[0]['gap_pts']:.1f} pts)")

       campus  cs_app  cs_adm  cs_rate  all_app  all_adm  all_rate  gap_pts
        Davis    4893     946     0.19   102988    45673      0.44     25.0
    San Diego   10785    2143     0.20   136727    38457      0.28      8.0
    Riverside    5068    4111     0.81    70863    61312      0.87      6.0
     Berkeley    9750     629     0.06   126830    14360      0.11      5.0
Santa Barbara    5865    1989     0.34   110173    42094      0.38      4.0
  Los Angeles    6803     498     0.07   145060    13659      0.09      2.0
       Irvine    6794    1874     0.28   124223    35658      0.29      1.0
   Santa Cruz    5952    4725     0.79    66393    48122      0.72     -7.0

Biggest CS penalty: Davis (25.0 pts)


## Q4. Fall 2025: IQR of admit GPA for Berkeley Computer Science
Uses `uc_freshman_admission_by_discipline.csv`, `admit_gpa_p25` / `admit_gpa_p75` columns.

In [8]:
row = disc[(disc.fall_term == 2025) & (disc.campus == 'Berkeley') & (disc.broad_discipline == 'Computer Science')]
p25 = row['admit_gpa_p25'].values[0]
p75 = row['admit_gpa_p75'].values[0]
print(f"p25={p25}, p75={p75}, IQR={p75 - p25:.2f}")

p25=4.2, p75=4.29, IQR=0.09


## Q5. Fall 2025: at how many of the 9 UC campuses was the White freshman admit rate higher than Hispanic/Latino(a)?
Uses `uc_admissions_summary_by_ethnicity.csv`, per-campus rows only (excludes `Systemwide`).

In [9]:
d = eth[(eth.fall_term == 2025) & (eth.entrant_level == 'freshman') & (eth.campus != 'Systemwide') &
        (eth.ethnicity.isin(['White', 'Hispanic/Latino(a)']))]

results = []
for campus in d.campus.unique():
    row = {'campus': campus}
    for e in ['White', 'Hispanic/Latino(a)']:
        app = d[(d.campus == campus) & (d.ethnicity == e) & (d.count_type == 'App')]['n'].values[0]
        adm = d[(d.campus == campus) & (d.ethnicity == e) & (d.count_type == 'Adm')]['n'].values[0]
        row[e] = adm / app
    results.append(row)

res = pd.DataFrame(results)
res['white_higher'] = res['White'] > res['Hispanic/Latino(a)']
print(res.to_string(index=False))
print(f"\nCampuses where White admit rate > Hispanic/Latino(a): {res['white_higher'].sum()} of {len(res)}")

       campus    White  Hispanic/Latino(a)  white_higher
     Berkeley 0.120194            0.118083          True
        Davis 0.450360            0.358684          True
       Irvine 0.274835            0.186305          True
  Los Angeles 0.099984            0.075304          True
       Merced 0.969137            0.950819          True
    Riverside 0.902254            0.832656          True
    San Diego 0.279440            0.259083          True
Santa Barbara 0.381941            0.308506          True
   Santa Cruz 0.791047            0.619164          True

Campuses where White admit rate > Hispanic/Latino(a): 9 of 9


## Q6. Systemwide, fall 2025: White vs. Hispanic/Latino(a) freshman admit rate — which is higher?
Same source, `campus == 'Systemwide'` this time.

In [10]:
d_sys = eth[(eth.fall_term == 2025) & (eth.entrant_level == 'freshman') & (eth.campus == 'Systemwide') &
            (eth.ethnicity.isin(['White', 'Hispanic/Latino(a)']))]
piv = d_sys.pivot_table(index='ethnicity', columns='count_type', values='n')
piv['admit_rate'] = piv['Adm'] / piv['App']
print(piv)
print("\nHigher systemwide admit rate:", piv['admit_rate'].idxmax())

count_type              Adm      App      Enr  admit_rate
ethnicity                                                
Hispanic/Latino(a)  41458.0  55624.0  14694.0    0.745326
White               26368.0  38390.0   8856.0    0.686846

Higher systemwide admit rate: Hispanic/Latino(a)


## Q7 (bonus). Class of 2023: share of Bay Area HS graduates who enrolled at a California Community College within 12 months
Uses `bay_area_modeling_table.csv`, `campus == 'Universitywide'`, `fall_term == 2023`, `enrolled_ccc` / `hs_completers`.

In [11]:
uw23 = bay[(bay.campus == 'Universitywide') & (bay.fall_term == 2023)]
tot_completers = uw23['hs_completers'].sum()
tot_ccc = uw23['enrolled_ccc'].sum()
print(f"CCC enrollment share, class of 2023: {tot_ccc:.0f} / {tot_completers:.0f} = {tot_ccc/tot_completers:.4f}")

CCC enrollment share, class of 2023: 21644 / 64345 = 0.3364


## Q8 (bonus). Mission San Jose HS, fall 2023: share of a-g completers who applied to at least one UC
`applicants` (Universitywide) ÷ `ag_completers`.

In [12]:
row = bay[(bay.high_school.str.contains('MISSION SAN JOSE', case=False, na=False)) &
          (bay.campus == 'Universitywide') & (bay.fall_term == 2023)]
share = row['applicants'].values[0] / row['ag_completers'].values[0]
print(f"applicants={row['applicants'].values[0]:.0f}, ag_completers={row['ag_completers'].values[0]:.0f}, share={share:.4f}")

applicants=420, ag_completers=424, share=0.9906


## Q9 (bonus, use with caution). Fall 2025: distinct Bay Area public high schools sending >=1 UC applicant
**Caveat:** `bay_area_modeling_table.csv` covers only the 9-county Bay Area, not all of California, and `school_type` has some missing labels for schools that are plausibly public charters (join gap in the source file). The cell below reports a few different cuts rather than one single number

In [14]:
uw25 = bay[(bay.campus == 'Universitywide') & (bay.fall_term == 2025) & (bay.applicants > 0)]

print("All school types, distinct high_school names:", uw25['high_school'].nunique())

pub_labeled = uw25[uw25.school_type.astype(str).str.contains('Public', na=False)]
print("Labeled 'Public' school_type only:", pub_labeled['high_school'].nunique())



All school types, distinct high_school names: 244
Labeled 'Public' school_type only: 209


## Q10 (bonus). Fall 2022-2025: which school MOST outperforms its expected UC Berkeley admit rate?
Controls for a-g completion rate, FRPM % (poverty), applicant GPA, and applicant pool size via linear regression; ranks schools by residual (actual − predicted admit rate).

**Caveat:** in-sample R^2 is low (~0.15) — treat this as directional, not precise. `applicant_gpa` also looks like it may be a merge artifact (identical value across years for a given school), so weigh that skepticism into any conclusion.

In [15]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

berk = bay[(bay.campus == 'Berkeley') & (bay.fall_term.between(2022, 2025))].copy()
feats = ['ag_completion_rate', 'frpm_pct', 'applicant_gpa', 'applicants']
target = 'admit_rate'

m = berk.dropna(subset=feats + [target]).copy()
X = m[feats]
y = m[target]

scaler = StandardScaler()
Xs = scaler.fit_transform(X)
lr = LinearRegression().fit(Xs, y)
print("R2 (in-sample):", lr.score(Xs, y))

m['predicted'] = lr.predict(Xs)
m['residual_pts'] = (m['admit_rate'] - m['predicted']) * 100

rank = m.groupby('high_school')['residual_pts'].mean().sort_values(ascending=False)
print(rank.head(10))

R2 (in-sample): 0.1454630354432236
high_school
ENVISION ACAD ARTS/TECHNOLOGY     34.126598
MISSION SENIOR HIGH SCHOOL        25.428256
RIO VISTA HIGH SCHOOL             24.778830
LUIS VALDEZ LEADERSHIP ACADEMY    12.704862
OAKLAND CHARTER HIGH SCHOOL       12.298577
SAINT HELENA HIGH SCHOOL          12.080658
TERRA NOVA HIGH SCHOOL            11.769847
JAMES LICK HIGH SCHOOL            10.823448
DOZIER-LIBBEY MEDICAL HIGH SCH    10.809316
HERCULES HIGH SCHOOL               9.608726
Name: residual_pts, dtype: float64
